# EDA
## FAOSTAT Producer Prices — Australia & New Zealand

**Assignment:** Data Visualisation Design & Storytelling — Parts 2 & 3  
**Narrative arc:** *The Sparkline* — What Is vs What Could Be  
**Stakeholder:** UN Food Systems Summit Panel  

---

**The Sparkline story in one sentence:**  
> *"When AUS/NZ commodity prices spike alongside the FFPI, the 15 most food-import-dependent,
hunger-vulnerable nations face the biggest crisis — and here is the gap between what is happening
and what targeted price stabilisation could achieve."*


---
## 0. Setup & Load All Cleaned Files


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Consistent plot style
plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 10,
})
COLORS = {'AUS': '#1d6fa4', 'NZL': '#e07b39',
          'FFPI': '#888888', 'danger': '#c0392b', 'safe': '#27ae60'}


In [ ]:
# Resolve paths from the repository root so the notebook works after clone.
def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'README.md').exists() and (candidate / 'data').exists():
            return candidate
    return start.resolve()

ROOT_DIR = find_repo_root(Path.cwd())
PROCESSED_DIR = ROOT_DIR / 'data' / 'processed'

# Load all processed files
master_usd = pd.read_csv(PROCESSED_DIR / 'master_producer_prices_usd.csv')
master_idx = pd.read_csv(PROCESSED_DIR / 'master_producer_price_index.csv')
ffpi_m     = pd.read_csv(PROCESSED_DIR / 'ffpi_monthly.csv', parse_dates=['date'])
ffpi_a     = pd.read_csv(PROCESSED_DIR / 'ffpi_annual.csv')
ghi        = pd.read_csv(PROCESSED_DIR / 'ghi_cleaned.csv')
wb         = pd.read_csv(PROCESSED_DIR / 'worldbank_food_import_pct.csv')

# Convenience splits
aus = master_usd[master_usd['iso3'] == 'AUS'].copy()
nzl = master_usd[master_usd['iso3'] == 'NZL'].copy()

print('All files loaded.')
print(f'Processed data directory: {PROCESSED_DIR}')
print(f'   master_usd : {master_usd.shape}')
print(f'   master_idx : {master_idx.shape}')
print(f'   ffpi_m     : {ffpi_m.shape}')
print(f'   ghi        : {ghi.shape}')
print(f'   worldbank  : {wb.shape}')


---
## 1. Dataset Overview & Structure


In [ ]:
print('MASTER USD — Shape & Columns')
print(f'Rows: {len(master_usd):,}   Columns: {master_usd.shape[1]}')
print()
print('Column dtypes:')
print(master_usd.dtypes.to_string())


In [ ]:
print('Countries and year coverage:')
cov = master_usd.groupby('country')['year'].agg(['min','max','count'])
cov.columns = ['first_year','last_year','n_observations']
print(cov.to_string())
print()
print(f'Total unique commodities: {master_usd["item"].nunique()}')
print(f'  AUS: {aus["item"].nunique()} commodities')
print(f'  NZL: {nzl["item"].nunique()} commodities')


In [ ]:
print('Month distribution (should all be Annual value for this download):')
print(master_usd['month'].value_counts())
print()
print('Flag breakdown:')
print(master_usd['flag'].value_counts())
print()
print('Imputed rows:', master_usd['is_imputed'].sum())
print('Outlier rows:', master_usd['is_outlier'].sum())


In [ ]:
# Statistical summary of producer prices
print('Descriptive Statistics — USD/tonne')
stats = master_usd.groupby('country')['value'].describe().round(2)
print(stats.to_string())


---
## 2. Missing Value Analysis

Understanding nulls is critical. `ghi_2025` being NaN for AUS/NZL is **by design** —
they are high-income nations not ranked by GHI. This is a feature, not a bug.


In [ ]:
null_counts = master_usd.isnull().sum()
null_pct    = (null_counts / len(master_usd) * 100).round(1)
null_df = pd.DataFrame({'null_count': null_counts, 'null_pct': null_pct})
null_df = null_df[null_df['null_count'] > 0]
print('Columns with missing values:')
print(null_df.to_string())
print()
print('Why ghi_2025 = NaN for all rows:')
print('  AUS and NZL are high-income nations not included in GHI rankings.')
print('  This is correct. The GHI data is used for the GLOBAL hunger context panel,')
print('  not for the AUS/NZL rows themselves.')
print()
print('ghi_cleaned.csv coverage:')
print(f'  Countries with GHI 2025 score: {ghi["ghi_2025"].notna().sum()}')
print(f'  Countries missing GHI 2025:   {ghi["ghi_2025"].isna().sum()} (use 2016 or 2008 as fallback)')


---
## 3. Commodity Price Analysis


In [ ]:
# Top 15 highest average priced commodities — AUS
top_aus = (aus.groupby('item')['value'].mean().sort_values(ascending=False).head(15))

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# AUS
axes[0].barh(top_aus.index[::-1], top_aus.values[::-1], color=COLORS['AUS'], alpha=0.85)
axes[0].set_xlabel('Avg Producer Price (USD/tonne)')
axes[0].set_title('Australia — Top 15 Commodities by Avg Price', fontweight='bold')
for i, v in enumerate(top_aus.values[::-1]):
    axes[0].text(v + 100, i, f'${v:,.0f}', va='center', fontsize=8)

# NZL
top_nzl = (nzl.groupby('item')['value'].mean().sort_values(ascending=False).head(15))
axes[1].barh(top_nzl.index[::-1], top_nzl.values[::-1], color=COLORS['NZL'], alpha=0.85)
axes[1].set_xlabel('Avg Producer Price (USD/tonne)')
axes[1].set_title('New Zealand — Top 15 Commodities by Avg Price', fontweight='bold')
for i, v in enumerate(top_nzl.values[::-1]):
    axes[1].text(v + 100, i, f'${v:,.0f}', va='center', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Price volatility — Coefficient of Variation (CV = std/mean) per commodity
cv_aus = aus.groupby('item')['value'].agg(['mean','std'])
cv_aus['cv'] = (cv_aus['std'] / cv_aus['mean'] * 100).round(1)
cv_aus = cv_aus.sort_values('cv', ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
top_cv = cv_aus.head(15)
bars = ax.barh(top_cv.index[::-1], top_cv['cv'].values[::-1], color=plt.cm.RdYlGn_r(top_cv['cv'].values[::-1]/top_cv['cv'].max()))
ax.set_xlabel('Coefficient of Variation (%) — Higher = More Volatile')
ax.set_title('Australia — Most Price-Volatile Commodities (1991–2024)', fontweight='bold')
ax.axvline(50, color='red', linestyle='--', alpha=0.5, label='50% CV threshold')
for i, (idx, row) in enumerate(top_cv.iloc[::-1].iterrows()):
    ax.text(row['cv'] + 0.5, i, f"{row['cv']:.0f}%", va='center', fontsize=8)
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
print('Most volatile commodities (CV > 50%):')
print(cv_aus[cv_aus['cv']>50][['mean','std','cv']].round(2).to_string())

---
## 4. Time Series Analysis — Price Trends & Crisis Years

Key global food price crises to annotate in the dashboard:
- **2007–2008:** Global food price crisis — wheat +61%, barley +82%, milk +50%
- **2010–2011:** Arab Spring — cereals spiked, political instability
- **2022:** Ukraine-Russia war — FFPI hit its highest level in history


In [ ]:
# Key commodities for time series
KEY_ITEMS = ['Wheat', 'Barley', 'Maize (corn)', 'Rice','Raw milk of cattle', 'Meat of cattle with the bone',
             'fresh or chilled','Soya beans', 'Canola (Rape or colza seed)']

# Filter — use whatever is available in AUS
available = [i for i in KEY_ITEMS if i in aus['item'].values]
if 'Rape or colza seed' in aus['item'].values:
    available.append('Rape or colza seed')
print('Available key items:', available)


In [ ]:
CRISIS_ZONES = [
    (2007, 2009, '#ffe5e5', '2007-09\nFood Crisis'),
    (2010, 2012, '#fff0e0', '2010-11\nArab Spring'),
    (2021, 2023, '#ffe5e5', '2022\nUkraine War'),
]

# Plot time series for 4 key staple commodities
plot_items = [i for i in ['Wheat','Barley','Maize (corn)','Rice','Soya beans',
              'Raw milk of cattle','Rape or colza seed','Sorghum'] if i in aus['item'].values][:4]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for ax, item in zip(axes, plot_items):
    aus_ts = aus[aus['item']==item].sort_values('year')
    nzl_ts = nzl[nzl['item']==item].sort_values('year')

    # Shade crisis zones
    for start, end, color, label in CRISIS_ZONES:
        ax.axvspan(start, end, alpha=0.3, color=color)
        ax.text((start+end)/2, ax.get_ylim()[1] if ax.get_ylim()[1]>0 else 1,
                label, ha='center', fontsize=7, color='#888')

    ax.plot(aus_ts['year'], aus_ts['value'], color=COLORS['AUS'],
            linewidth=2, label='Australia', marker='o', markersize=3)
    if len(nzl_ts) > 0:
        ax.plot(nzl_ts['year'], nzl_ts['value'], color=COLORS['NZL'],
                linewidth=2, label='New Zealand', linestyle='--', marker='s', markersize=3)

    # Annotate outliers
    out = aus_ts[aus_ts['is_outlier']]
    if len(out) > 0:
        ax.scatter(out['year'], out['value'], color=COLORS['danger'],
                   zorder=5, s=60, label='Outlier flagged')

    ax.set_title(f'{item}', fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_ylabel('USD / tonne')
    ax.legend(fontsize=8)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))

plt.suptitle('Producer Price Time Series — Key Staple Commodities', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# Year-on-Year % change for key items
print('Year-on-Year Price Changes for Key Commodities (AUS):')
for item in [i for i in ['Wheat','Barley','Maize (corn)','Raw milk of cattle',
                          'Meat of cattle with the bone, fresh or chilled','Soya beans']
             if i in aus['item'].values]:
    sub = aus[aus['item']==item].sort_values('year')[['year','value']].copy()
    sub['yoy_pct'] = sub['value'].pct_change() * 100
    peak = sub.nlargest(1,'yoy_pct')
    trough = sub.nsmallest(1,'yoy_pct')
    print(f'{item[:45]:<45}')
    print(f'  Peak increase : +{peak["yoy_pct"].values[0]:.1f}% in {peak["year"].values[0]}')
    print(f'  Peak decrease :  {trough["yoy_pct"].values[0]:.1f}% in {trough["year"].values[0]}')


---
## 5. Enrichment 1 — FAO Food Price Index (FFPI)

**How it connects:** Joined to FAOSTAT on `year`. Every row in `master_usd` now has
the global FFPI score for that year. This enables side-by-side comparison:
*'Did AUS commodity prices move with, against, or ahead of the global index?'*

**Dashboard use:**
- Dual-axis line chart: AUS/NZ price index vs FFPI on same axes
- Monthly FFPI used for spike annotations and scrollytelling narrative
- Crisis year callout boxes triggered when FFPI > 120 (above base)


In [ ]:
# FFPI all indices over time — the global shock landscape
fig, ax = plt.subplots(figsize=(14, 6))

ffpi_plot = ffpi_a[ffpi_a['year'] >= 1991].copy()
colors_ffpi = {'ffpi_food':'#2c3e50','ffpi_cereals':'#e74c3c',
               'ffpi_meat':'#8e44ad','ffpi_dairy':'#2980b9',
               'ffpi_oils':'#f39c12','ffpi_sugar':'#27ae60'}

for col, color in colors_ffpi.items():
    ax.plot(ffpi_plot['year'], ffpi_plot[col], label=col.replace('ffpi_','').title(),
            color=color, linewidth=1.8, alpha=0.9)

# Shade crisis periods
ax.axvspan(2007, 2009, alpha=0.12, color='red', label='_nolegend_')
ax.axvspan(2010, 2012, alpha=0.12, color='orange', label='_nolegend_')
ax.axvspan(2021, 2023, alpha=0.12, color='red', label='_nolegend_')
ax.axhline(100, color='black', linewidth=0.8, linestyle='--', alpha=0.5, label='Base (2014-2016=100)')

# Annotate peaks
ax.annotate('2008\nFood Crisis', xy=(2008, 180), fontsize=8, color='red',
            xytext=(2005, 190), arrowprops=dict(arrowstyle='->', color='red'))
ax.annotate('2022\nUkraine War\n(All-time high)', xy=(2022, 187), fontsize=8, color='red',
            xytext=(2018, 200), arrowprops=dict(arrowstyle='->', color='red'))

ax.set_xlabel('Year')
ax.set_ylabel('Price Index (2014-2016 = 100)')
ax.set_title('FAO Food Price Index — All Sub-Indices (1991-2025)', fontweight='bold')
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# AUS Price Index vs FFPI cereals — does AUS track the global cereals market?
aus_idx = master_idx[master_idx['iso3']=='AUS'].groupby('year')['value'].mean().reset_index()
nzl_idx = master_idx[master_idx['iso3']=='NZL'].groupby('year')['value'].mean().reset_index()
merged = aus_idx.merge(ffpi_a[['year','ffpi_food']], on='year', how='inner')

fig, ax1 = plt.subplots(figsize=(14, 6))
ax2 = ax1.twinx()

ax1.plot(aus_idx['year'], aus_idx['value'], color=COLORS['AUS'],
         linewidth=2.5, label='AUS Avg Price Index')
ax1.plot(nzl_idx['year'], nzl_idx['value'], color=COLORS['NZL'],
         linewidth=2.5, linestyle='--', label='NZL Avg Price Index')
ax2.plot(merged['year'], merged['ffpi_food'], color=COLORS['FFPI'],
         linewidth=1.5, linestyle=':', label='FFPI Food (global)', alpha=0.7)

for start, end, color, label in CRISIS_ZONES:
    ax1.axvspan(start, end, alpha=0.15, color=color)

ax1.set_xlabel('Year')
ax1.set_ylabel('AUS/NZL Producer Price Index (2014-2016=100)', color=COLORS['AUS'])
ax2.set_ylabel('FFPI Food Index (2014-2016=100)', color=COLORS['FFPI'])
ax1.set_title('AUS & NZL Producer Price Index vs Global FFPI — Enrichment Join in Action',fontweight='bold')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labels1+labels2, loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()
print('Correlation — AUS Price Index vs FFPI Food:')
print(f'  Pearson r = {merged["value"].corr(merged["ffpi_food"]):.3f}')
print('  (>0.7 = strong positive relationship — AUS tracks global prices closely)')


---
## 6. Enrichment 2 — Global Hunger Index (GHI)

**How it connects:** Joined on `iso3`. AUS/NZL get `NaN` (correct — not in ranking).
The full GHI dataset is used for the global hunger context panel.

**Dashboard use:**
- World choropleth coloured by GHI 2025 score
- When user selects a country on the map → tooltip shows GHI trend 2000→2025
- Narrative: *'These countries improved — but they remain exposed to AUS/NZ price shocks'*


In [ ]:
# GHI distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of GHI 2025
ghi_valid = ghi.dropna(subset=['ghi_2025'])
axes[0].hist(ghi_valid['ghi_2025'], bins=20, color='#e74c3c', alpha=0.8, edgecolor='white')
axes[0].axvline(ghi_valid['ghi_2025'].mean(), color='black', linestyle='--',label=f'Mean: {ghi_valid["ghi_2025"].mean():.1f}')
axes[0].set_xlabel('GHI Score 2025')
axes[0].set_ylabel('Number of Countries')
axes[0].set_title('Distribution of Hunger Scores Globally (2025)', fontweight='bold')
axes[0].legend()

# GHI change 2000 → 2025 (improvement vs worsening)
ghi_change = ghi.dropna(subset=['ghi_2000','ghi_2025']).copy()
ghi_change['change'] = ghi_change['ghi_2025'] - ghi_change['ghi_2000']
ghi_change = ghi_change.sort_values('change')
improved = ghi_change[ghi_change['change'] < 0]
worsened = ghi_change[ghi_change['change'] > 0]
axes[1].barh(range(len(ghi_change)), ghi_change['change'],color=['#27ae60' if x < 0 else '#e74c3c' for x in ghi_change['change']],alpha=0.8)
axes[1].axvline(0, color='black', linewidth=1)
axes[1].set_xlabel('GHI Score Change (2000 → 2025)')
axes[1].set_title(f'Hunger Progress: {len(improved)} Improved, {len(worsened)} Worsened', fontweight='bold')
axes[1].set_yticks([])
green_patch = mpatches.Patch(color='#27ae60', label='Improved (lower score)')
red_patch   = mpatches.Patch(color='#e74c3c', label='Worsened (higher score)')
axes[1].legend(handles=[green_patch, red_patch])

plt.tight_layout()
plt.show()


In [ ]:
# Top 15 most food-insecure countries
top_hunger = ghi.nlargest(15, 'ghi_2025')[['country_ghi','ghi_2025','ghi_2016','ghi_2008','ghi_2000']]
fig, ax = plt.subplots(figsize=(13, 6))

x = np.arange(len(top_hunger))
w = 0.2
for i, (col, label, color) in enumerate([
    ('ghi_2000','2000','#95a5a6'),
    ('ghi_2008','2008','#e67e22'),
    ('ghi_2016','2016','#e74c3c'),
    ('ghi_2025','2025','#8e44ad'),
]):
    vals = top_hunger[col].values
    ax.bar(x + i*w, vals, w, label=label, color=color, alpha=0.85)

ax.set_xticks(x + 1.5*w)
ax.set_xticklabels(top_hunger['country_ghi'], rotation=35, ha='right', fontsize=8)
ax.set_ylabel('GHI Score (higher = worse hunger)')
ax.set_title('Top 15 Most Food-Insecure Countries — GHI Trend (2000–2025)', fontweight='bold')
ax.legend(title='Year')
plt.tight_layout()
plt.show()


---
## 7. Enrichment 3 — World Bank Food Import Dependency

**How it connects:** Joined on `iso3 + year`. For AUS/NZL rows, `food_import_pct` shows
how much of their own merchandise imports is food — they are both low (exporter nations).

**Dashboard use:**
- Scatter plot: food_import_pct (y) vs GHI score (x) — vulnerability quadrant
- What-if slider: user selects 'price increase %' → highlights countries whose food bill
  would exceed a threshold
- Area chart: AUS/NZ food import % over time (they are NET exporters)


In [ ]:
# AUS vs NZL food import dependency over time
fig, ax = plt.subplots(figsize=(13, 5))

for iso, label, color in [('AUS','Australia',COLORS['AUS']),('NZL','New Zealand',COLORS['NZL'])]:
    sub = wb[wb['iso3']==iso].sort_values('year')
    ax.plot(sub['year'], sub['food_import_pct'], color=color,
            linewidth=2.5, label=label, marker='o', markersize=4)
    ax.fill_between(sub['year'], sub['food_import_pct'], alpha=0.1, color=color)

ax.set_xlabel('Year')
ax.set_ylabel('Food Imports (% of Merchandise Imports)')
ax.set_title('AUS & NZL: Food Import Dependency Over Time\n(Low % = net food exporters)', fontweight='bold')
ax.legend()
ax.annotate('AUS: ~5-8%\n(Net exporter)', xy=(2015, 5.5), fontsize=9, bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))
ax.annotate('NZL: ~10-13%\n(More import-\ndependent)', xy=(2016, 11), fontsize=9,bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.5))
plt.tight_layout()
plt.show()


In [ ]:
# Global distribution of food import dependency (latest year)
wb_latest = (
    wb.sort_values('year', ascending=False)
    .groupby('iso3')
    .first()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(13, 5))
ax.hist(wb_latest['food_import_pct'], bins=40, color='#3498db', alpha=0.8, edgecolor='white')
ax.axvline(wb_latest['food_import_pct'].mean(), color='black', linestyle='--',
           label=f'Mean: {wb_latest["food_import_pct"].mean():.1f}%')
ax.axvline(wb_latest[wb_latest['iso3']=='AUS']['food_import_pct'].values[0],
           color=COLORS['AUS'], linewidth=2,
           label=f'Australia: {wb_latest[wb_latest["iso3"]=="AUS"]["food_import_pct"].values[0]:.1f}%')
ax.axvline(wb_latest[wb_latest['iso3']=='NZL']['food_import_pct'].values[0],
           color=COLORS['NZL'], linewidth=2, linestyle='--',
           label=f'New Zealand: {wb_latest[wb_latest["iso3"]=="NZL"]["food_import_pct"].values[0]:.1f}%')
ax.set_xlabel('Food Import % (latest year)')
ax.set_ylabel('Number of Countries')
ax.set_title('Global Distribution of Food Import Dependency', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()


---
## 8.Vulnerability Matrix — The Core Dashboard Insight

This is the analytical centrepiece of the Sparkline narrative.
By joining GHI (hunger severity) × World Bank (food import dependency),
we can rank countries by their composite vulnerability to AUS/NZ price shocks.

**`vulnerability score = ghi_2025 × food_import_pct`**

Countries in the top-right quadrant (high hunger + high import dependency) are
the most exposed when AUS/NZ commodity prices rise.


In [ ]:
# Build vulnerability matrix
ghi_slim   = ghi[['iso3','country_ghi','ghi_2025']].dropna(subset=['iso3','ghi_2025'])
wb_latest2 = wb.sort_values('year',ascending=False).groupby('iso3').first().reset_index()[['iso3','food_import_pct']]
vuln = ghi_slim.merge(wb_latest2, on='iso3', how='inner')
vuln['vulnerability_score'] = vuln['ghi_2025'] * vuln['food_import_pct']
vuln = vuln.sort_values('vulnerability_score', ascending=False)

print('Top 15 most vulnerable countries to AUS/NZ food price shocks:')
print(vuln.head(15)[['country_ghi','ghi_2025','food_import_pct','vulnerability_score']].round(2).to_string(index=False))


In [ ]:
# Scatter plot — the vulnerability quadrant (DASHBOARD VISUAL 3)
fig, ax = plt.subplots(figsize=(13, 8))

# Quadrant medians
ghi_med  = vuln['ghi_2025'].median()
imp_med  = vuln['food_import_pct'].median()

scatter = ax.scatter(
    vuln['ghi_2025'],
    vuln['food_import_pct'],
    c=vuln['vulnerability_score'],
    cmap='RdYlGn_r',
    s=80, alpha=0.85, edgecolors='white', linewidths=0.5
)

# Quadrant lines
ax.axvline(ghi_med, color='grey', linestyle='--', alpha=0.5)
ax.axhline(imp_med, color='grey', linestyle='--', alpha=0.5)

# Label quadrants
ax.text(2, vuln['food_import_pct'].max()*0.92, 'LOW HUNGER\nHIGH IMPORT DEPENDENCY',
        fontsize=8, color='#3498db', alpha=0.7)
ax.text(ghi_med+0.5, vuln['food_import_pct'].max()*0.92, '⚠️ HIGH HUNGER\nHIGH IMPORT DEPENDENCY\n(MOST VULNERABLE)',
        fontsize=8, color='#c0392b', fontweight='bold')

# Label top 10 most vulnerable
for _, row in vuln.head(10).iterrows():
    ax.annotate(row['country_ghi'], xy=(row['ghi_2025'], row['food_import_pct']),
                fontsize=7, xytext=(3, 3), textcoords='offset points')

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Vulnerability Score (GHI × Import %)', fontsize=9)

ax.set_xlabel('GHI Score 2025 (Higher = More Hunger)', fontsize=11)
ax.set_ylabel('Food Import % of Merchandise Imports (Latest)', fontsize=11)
ax.set_title('Global Vulnerability Matrix\n'
             'Countries Most Exposed to AUS/NZ Food Price Shocks', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


---
## 9. What-If Scenario Analysis — Dashboard Advanced Feature

This powers the **what-if parameterisation** slider in the dashboard.
The logic: if AUS/NZ producer prices rise by X%, countries that import food
will see their food import bill rise proportionally.

**Formula:** `implied_food_cost_increase = food_import_pct × price_shock_pct / 100`


In [ ]:
def what_if_analysis(price_shock_pct: float) -> pd.DataFrame:
    df = vuln.copy()
    df['implied_cost_increase_pct'] = df['food_import_pct'] * price_shock_pct / 100
    df['scenario_label'] = f'+{price_shock_pct}% price shock'
    return df.sort_values('implied_cost_increase_pct', ascending=False)

# Show three scenarios
fig, axes = plt.subplots(1, 3, figsize=(18, 7))

for ax, shock in zip(axes, [10, 20, 40]):
    result = what_if_analysis(shock)
    top = result.head(12)
    colors = ['#c0392b' if g > 25 else '#e67e22' if g > 15 else '#f1c40f'
              for g in top['ghi_2025']]
    ax.barh(top['country_ghi'][::-1], top['implied_cost_increase_pct'][::-1],color=colors[::-1], alpha=0.9)
    ax.set_xlabel('Implied Food Cost Increase (%)')
    ax.set_title(f'Scenario: +{shock}% AUS/NZ\nProducer Price Shock', fontweight='bold')
    for i, (_, row) in enumerate(top.iloc[::-1].iterrows()):
        ax.text(row['implied_cost_increase_pct']+0.1, i,
                f"{row['implied_cost_increase_pct']:.1f}%", va='center', fontsize=7)

red_p    = mpatches.Patch(color='#c0392b', label='GHI > 25 (Alarming)')
orange_p = mpatches.Patch(color='#e67e22', label='GHI 15-25 (Serious)')
yellow_p = mpatches.Patch(color='#f1c40f', label='GHI < 15 (Moderate)')
axes[2].legend(handles=[red_p,orange_p,yellow_p], loc='lower right', fontsize=8)

plt.suptitle('What-If: Impact of AUS/NZ Food Price Shocks on Globally Vulnerable Nations',fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


---
## 10. Monthly FFPI — Narrative Scrollytelling Data

The monthly FFPI is used for the **narrative scrollytelling** feature in the dashboard.
As the user scrolls, annotations appear at key spikes explaining *why* prices rose.


In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

ffpi_recent = ffpi_m[ffpi_m['year'] >= 2000].copy()
ax.plot(ffpi_recent['date'], ffpi_recent['ffpi_food'], color='#2c3e50', linewidth=1.8, label='FAO Food Price Index')
ax.fill_between(ffpi_recent['date'], 100, ffpi_recent['ffpi_food'],where=ffpi_recent['ffpi_food'] > 100,alpha=0.2, color='#e74c3c', label='Above base (inflationary)')
ax.fill_between(ffpi_recent['date'], 100, ffpi_recent['ffpi_food'],where=ffpi_recent['ffpi_food'] <= 100, alpha=0.15, color='#27ae60', label='Below base (deflationary)')

# Key event annotations
events = [
    ('2008-06', 'Global Food\nCrisis Peak'),
    ('2011-02', 'Arab Spring\nFood Protests'),
    ('2016-01', 'Base period\nstart'),
    ('2022-03', '2022: All-time\nhigh — Ukraine'),
]
for date_str, label in events:
    date = pd.to_datetime(date_str)
    val  = ffpi_recent[ffpi_recent['date']==date]['ffpi_food']
    if len(val) == 0:
        val = ffpi_recent[ffpi_recent['date'] >= date]['ffpi_food'].values[0]
        date = ffpi_recent[ffpi_recent['date'] >= pd.to_datetime(date_str)]['date'].values[0]
        date = pd.Timestamp(date)
    else:
        val = val.values[0]
    ax.annotate(label, xy=(date, val), fontsize=7.5, color='#c0392b',
                xytext=(0, 20), textcoords='offset points',
                arrowprops=dict(arrowstyle='->', color='#c0392b', lw=1))

ax.axhline(100, color='black', linewidth=0.8, linestyle='--', alpha=0.4, label='Base = 100')
ax.set_xlabel('Date')
ax.set_ylabel('FFPI (2014-2016 = 100)')
ax.set_title('Monthly FAO Food Price Index (2000–2026) — Scrollytelling Timeline Data',fontweight='bold')
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()


---
## 11. Summary — Visual Plan for Dashboard (Architects' Handoff)

Based on this EDA, here is the recommended visual plan for Rishi & Manh (Architects):

### Visual 1 — Time Series Line Chart (Temporal)
**File:** `master_producer_prices_usd.csv` + `ffpi_annual.csv`  
**X:** Year | **Y-left:** AUS/NZL producer price (USD/tonne) | **Y-right:** FFPI Food Index  
**Advanced feature:** Context-aware filter — select commodity → updates all other visuals  
**Story:** *'AUS prices track global shocks — the 2022 spike hit both local farmers and global markets'*

### Visual 2 — Price Volatility Bar/Heatmap (Comparative)
**File:** `master_producer_price_index.csv`  
**X:** Commodity | **Y:** Avg Price Index | **Colour:** CV (coefficient of variation)  
**Advanced feature:** Tooltip with mini sparkline chart per commodity  
**Story:** *'Some commodities are far more volatile than others — here is the risk landscape'*

### Visual 3 — Vulnerability Scatter Plot (Enrichment centrepiece)
**File:** `ghi_cleaned.csv` + `worldbank_food_import_pct.csv`  
**X:** GHI Score | **Y:** Food Import % | **Size/Colour:** Vulnerability score  
**Advanced feature:** Hover tooltip shows country name, GHI trend, top imported commodities  
**Story:** *'Haiti, Niger, Somalia — these countries cannot absorb another price shock'*

### Visual 4 — What-If Scenario Panel (Call to action)
**File:** `master_producer_prices_usd.csv` + vulnerability matrix  
**Control:** Slider (price shock %) + dropdown (commodity)  
**Output:** Bar chart of most impacted countries + implied cost increase  
**Advanced feature:** What-if parameterisation  
**Story:** *'If prices stabilise to 2019 levels, 12 countries exit the extreme-risk zone'*

---
*EDA authored by: Chaitya Nanavati & Shreyas Yadugani (Analysts)*  
*Hand off `master_producer_prices_usd.csv`, `ghi_cleaned.csv`, `worldbank_food_import_pct.csv`, `ffpi_monthly.csv` to Architects.*


In [ ]:
# Final summary stats
print('PIPELINE OUTPUT SUMMARY')
print(f'Primary dataset rows    : {len(master_usd):,}')
print(f'Commodities (AUS)       : {aus["item"].nunique()}')
print(f'Commodities (NZL)       : {nzl["item"].nunique()}')
print(f'Year range              : {master_usd["year"].min()} – {master_usd["year"].max()}')
print(f'Outliers flagged        : {master_usd["is_outlier"].sum()}')
print(f'Imputed rows            : {master_usd["is_imputed"].sum()}')
print(f'FFPI annual records     : {len(ffpi_a)}')
print(f'FFPI monthly records    : {len(ffpi_m)}')
print(f'GHI countries           : {ghi["ghi_2025"].notna().sum()}')
print(f'World Bank rows         : {len(wb):,}')
print(f'Vulnerability matrix    : {len(vuln)} countries scored')
print()
print('Top 5 most vulnerable countries to AUS/NZ price shocks:')
print(vuln.head(5)[['country_ghi','ghi_2025','food_import_pct','vulnerability_score']].round(2).to_string(index=False))
